# ***Parallel Chain***

In [2]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()  # This loads the variables from .env

True

# Chain With Parallel Chains

In [3]:
# Task -1 [Prompt]
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie summarizer."),
    ("human", "Please summarize the movie in brief: {input}"),
])

In [4]:
# Task -2 [LLM]

llm_openai = ChatOpenAI(model_name="gpt-5-mini", temperature=0, openai_api_key=os.environ.get("OPENAI_API_KEY"))

In [5]:
# Task -3 [String Parser]

str_parser = StrOutputParser()

In [6]:
# Task -4 [Custom Runnable]
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text: str) -> dict:
    return {"text": text}

# Create a RunnableLambda for the dictionary_maker function
dictionary_maker_runnable = RunnableLambda(dictionary_maker)

### Parallel Chain 1

In [7]:
# TASK -1 [Prompt]
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Linkedin post generator."),
    ("human", "Please generate the Linkedin post in brief: {text}"),       
])

# TASK -2 [LLM]
linkedin_llm = ChatOpenAI(model_name="gpt-5-mini", temperature=0, openai_api_key=os.environ.get("OPENAI_API_KEY"))

# TASK -3 [Chain]
str_parser_post = StrOutputParser()

chain_linkedin = linkedin_prompt | linkedin_llm | str_parser_post

### Parallel Chain 2

In [10]:
def insta_chain(text:dict)->dict:

    text = text["text"]

    # TASK -1 [Prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a Instagram post generator."),
        ("human", "Please generate the Instagram post in brief: {text}"),       
    ])

    # TASK -2 [LLM]
    insta_llm = ChatOpenAI(model_name="gpt-5-mini", temperature=0, openai_api_key=os.environ.get("OPENAI_API_KEY"))

    # TASK -3 [Chain]
    str_parser_post = StrOutputParser() 

    chain_insta = insta_prompt | insta_llm | str_parser_post
    
    post = chain_insta.invoke(text)
    
    return post

chain_instagram = RunnableLambda(insta_chain)


### Final Orchestrator

In [16]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

final_chain = (
    prompt_template
    | llm_openai
    | str_parser
    | dictionary_maker_runnable
    | RunnableParallel(branches = {"linkedIn" : chain_linkedin, "instagram" : chain_instagram})
    
)

In [17]:
final_chain.invoke("KGF")

{'branches': {'linkedIn': 'KGF follows Rocky, a fiercely ambitious young man who rises from poverty to dominate the brutal Kolar Gold Fields, toppling corrupt powers and becoming a feared — sometimes messianic — figure among oppressed miners. Stylized and violent, the films explore ambition, class struggle and how power can corrupt, with Chapter 1 ending on a cliffhanger leading into Chapter 2. A powerful story about grit, consequences and leadership — and a reminder that ambition without responsibility can be a double-edged sword. #KGF #Leadership #Ambition #Storytelling',
  'instagram': 'From Mumbai streets to the blood-soaked Kolar Gold Fields — Rocky’s rise is raw, ruthless and larger-than-life. A stylish, violent epic about ambition, class and the cost of power. Chapter 1 ends on a cliffhanger — ready for Chapter 2? 🎬🔥\n\nDrop a 🔥 if you’re Team Rocky.  \n#KGF #Rocky #KGFChapter1 #KGFChapter2 #KannadaCinema #ActionDrama #Ambition #Power'}}

## Chain as a Runnable

In [18]:
# TASk -1 [Beautify Function]

def beautify_function(final_response: dict) -> str:
    linked_response_text = final_response['branches']['linkedIn']
    instagram_response_text = final_response['branches']['instagram']   
    return {"linkedIn": linked_response_text, "instagram": instagram_response_text} 

beautify_runnable = RunnableLambda(beautify_function)

# TASK -2 [Final Chain]

# final_chain


# beautify_chain    
beautify_chain = final_chain | beautify_runnable

beautify_chain.invoke("Pushpa")
    


{'linkedIn': 'Just watched Pushpa: The Rise — a raw Telugu action-drama about Pushpa Raj (Allu Arjun), a lowly laborer who claws his way up the red sandalwood smuggling network, building power through ambition, ruthlessness and charisma while romancing Srivalli (Rashmika Mandanna) and sparring with a determined cop (Fahadh Faasil). Gritty action, earthy dialogue and punchy music drive a violent rise that ends on a cliffhanger — a provocative take on ambition, power and moral compromise. Who else has seen it and what leadership or ethical lessons did you take away? #Pushpa #AlluArjun #Cinema #Leadership',
 'instagram': 'Pushpa Raj rises from a lowly labourer to a ruthless kingpin in this gritty Telugu action-drama — power, betrayal, raw action and a cliffhanger that begs for a sequel. 🔥🌲\n\nStarring: Allu Arjun, Rashmika Mandanna, Fahadh Faasil. \nSeen it? Drop your favourite scene below! 👇\n\n#PushpaTheRise #Pushpa #AlluArjun #RashmikaMandanna #FahadhFaasil #Tollywood #ActionDrama #Cli